# B02: Unit-aware Program of Thoughts (PoT)
Baseline for Dataset Type 2 (Physics) using `pint` to evaluate and generate code.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import pint
import re

def find_repo_root(start: Path | None = None) -> Path:
    """Return the repository root by searching for pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Could not find pyproject.toml.")

ROOT = find_repo_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from common.config import TYPE2_PATH
from common.utils import load_physics_dataset

# Initialize the unit registry
ureg = pint.UnitRegistry()
Q_ = ureg.Quantity

### 1. Data Loading
Use data loading logic from `common.utils` and configuration from `common.config`.

In [2]:
# Load physics dataset using shared function
df_physics = load_physics_dataset(TYPE2_PATH)
print(f"Loaded {len(df_physics)} physics problems.")
df_physics.head()

Loaded 1755 physics problems.


,id,group_id,task_type,question_type,question,premises_nl,premises_fol,support_idx,gold_answer,gold_unit,gold_explanation,source_path,id_prefix,answer_type,stratify_label
0,physics_TD401,physics_TD401,physics,physics,Calculate the energy stored in capacitor C whe...,[],[],[],45,J,Step 1: Identify the given values for capacita...,/mnt/shared_data/workspace/ura/Exact2026/src/d...,TD,numeric_with_unit,physics::TD::numeric_with_unit
1,physics_TD402,physics_TD402,physics,physics,"Calculate the capacitance C of the capacitor, ...",[],[],[],100,μF,Step 1: Identify the given values from the que...,/mnt/shared_data/workspace/ura/Exact2026/src/d...,TD,numeric_with_unit,physics::TD::numeric_with_unit
2,physics_LD001,physics_LD001,physics,physics,"Two charges, q1 = 6 × 10^-8 C and q2 = -6 × 10...",[],[],[],0.05,N,Step 1: Identify the given charges and distanc...,/mnt/shared_data/workspace/ura/Exact2026/src/d...,LD,numeric_with_unit,physics::LD::numeric_with_unit
3,physics_LD002,physics_LD002,physics,physics,Three electric charges are placed at three fix...,[],[],[],24.45 × 10^-3,N,Step 1: Identify the charges and distances giv...,/mnt/shared_data/workspace/ura/Exact2026/src/d...,LD,numeric_with_unit,physics::LD::numeric_with_unit
4,physics_LD003,physics_LD003,physics,physics,Points A and B are separated by 20 cm in air. ...,[],[],[],6.76,N,Step 1: Identify the given charges and distanc...,/mnt/shared_data/workspace/ura/Exact2026/src/d...,LD,numeric_with_unit,physics::LD::numeric_with_unit


### 2. Prompting (Unit-aware PoT)
Design a prompt to ask the model to generate Python code using `pint`.

In [3]:
from common.prompts import PoTPromptInterface, get_prompt_builder

### 3. Code Execution (Sandbox)
Extract Python code from the LLM response and execute it safely.

In [4]:
import func_timeout

def extract_python_code(response: str) -> str:
    """
    Extracts the Python code snippet from the LLM's full text response.
    It uses Regex to find the content specifically enclosed within ```python ... ``` tags.
    Returns None if no python code block is found, allowing the system to trigger a retry.
    """
    match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
    if match:
        return match.group(1)
    return None

def execute_pot_code(code_str: str):
    """
    A secure Sandbox to execute the Python code generated by the LLM, 
    closely mimicking the `safe_execute` mechanism from the original POT paper.
    
    - Security 1 (Namespace): Only injects the `pint` library into the execution environment, 
      preventing the LLM from calling dangerous system functions.
    - Security 2 (Timeout): Uses `func_timeout` to force the process to stop after 5 seconds, 
      protecting the system from infinite loops generated by the LLM.
    """
    if code_str is None:
        return "Error: No code provided to execute."
        
    def execute(x):
        namespace = {'pint': pint, 'ureg': ureg, 'Q_': Q_}
        try:
            exec(x, namespace)
            # Retrieve the value of the 'ans' variable as dictated by the POT paper
            if 'ans' in namespace:
                return namespace['ans']
            else:
                return "Error: 'ans' variable not found."
        except Exception as e:
            return f"Error: {e}"
            
    try:
        # The POT paper uses func_timeout with a 5-second limit
        ans = func_timeout.func_timeout(5.0, execute, args=(code_str,))
    except func_timeout.FunctionTimedOut:
        ans = "Error: Execution timed out (5s)."
    except Exception as e:
        ans = f"Error: {e}"

    return ans

### 4. Inference Loop
Connect Prompt -> LLM API -> Code Execution

In [ ]:
from collections import Counter

def majority_vote(predictions: list) -> any:
    """
    Implements the Self-Consistency (Majority Voting) decoding strategy.
    Takes a list of predictions (from multiple LLM generation paths),
    filters out errors, and returns the most frequently occurring answer.
    """
    valid_preds = [p for p in predictions if not isinstance(p, str) or not p.startswith("Error")]
    if not valid_preds:
        return predictions[0] if predictions else "Error: No predictions"
    
    # Simple exact match voting. For floats, we might need rounding in a real scenario.
    vote_counts = Counter(valid_preds)
    return vote_counts.most_common(1)[0][0]

def run_baseline_2(df: pd.DataFrame, num_samples: int = 5, strategy: str = 'greedy', num_paths: int = 1, prompt_builder: PoTPromptInterface = None, max_retries: int = 1):
    """
    The main Inference Loop for Baseline B02.
    
    Decoding Strategy Configuration:
    - strategy = 'greedy': (Default) The LLM generates exactly 1 code path (fast, cost-effective).
    - strategy = 'self_consistency': The LLM generates `num_paths` different code paths for the same 
      question (requires Temperature > 0). Runs all paths and takes the majority vote (slower, but highly accurate).
    
    Prompt Configuration:
    - prompt_builder: Allows injecting ZeroShotPoTPrompt, OneShotPoTPrompt, or EightShotPoTPrompt.
      If None, defaults to OneShotPoTPrompt.
      
    Fallback Mechanism:
    - max_retries: If the LLM response doesn't contain valid code block (```python ... ```), retry the prompt.
    """
    results = []
    
    if prompt_builder is None:
        prompt_builder = get_prompt_builder('one_shot')
        
    if strategy == 'greedy':
        num_paths = 1

    for idx, row in df.head(num_samples).iterrows():
        question = row['question']
        expected_answer = row['gold_answer']
        unit = row['gold_unit']
        
        # Use the injected prompt builder interface to generate the prompt
        prompt = prompt_builder.build_prompt(question, unit)
        
        # =====================================================================
        # INTEGRATION POINT: REPLACE THIS MOCK WITH REAL LLM API CALL
        # =====================================================================
        # To run this for real,import your LLM client 
        #  
        # 
        # Example for real integration with retry logic:
        # path_predictions = []
        # for _ in range(num_paths):
        #     for attempt in range(max_retries + 1):
        #         response = call_llm(prompt, temperature=(0.0 if strategy == 'greedy' else 0.5))
        #         code = extract_python_code(response)
        #         if code is not None:
        #             break # Success, we got code!
        #         print(f"Warning: No code found on attempt {attempt+1}. Retrying...")
        #     pred = execute_pot_code(code)
        #     path_predictions.append(pred)
        # =====================================================================
        
        # Simulated LLM responses for testing the pipeline locally
        # We simulate the process of extracting code and executing it.
        path_predictions = []
        for _ in range(num_paths):
            # Mocking the retry logic
            for attempt in range(max_retries + 1):
                # Let's mock a scenario where the first attempt returns no code, 
                # but the second attempt (retry) succeeds.
                if attempt == 0:
                    mock_response = "I am sorry, but as an AI, I cannot calculate this. Here is some text but no code."
                else:
                    mock_response = """```python\nans = 45.0\n```"""
                
                code = extract_python_code(mock_response)
                if code is not None:
                    break # Successfully found python code
                print(f"ID {row['id']}: No code found on attempt {attempt+1}. Retrying...")
            
            pred = execute_pot_code(code)
            path_predictions.append(pred)
            
        final_prediction = majority_vote(path_predictions) if strategy == 'self_consistency' else path_predictions[0]
        
        results.append({
            'id': row['id'],
            'question': question,
            'expected': expected_answer,
            'prediction': final_prediction,
            'all_paths': path_predictions,
            'unit': unit
        })
        
    return pd.DataFrame(results)

### 5. Execute
Run the mock inference loop to test the pipeline.

In [6]:
# Run the mock loop
df_results = run_baseline_2(df_physics, num_samples=3)

# Display results
df_results

,id,question,expected,prediction,all_paths,unit
0,physics_TD401,Calculate the energy stored in capacitor C whe...,45,45.0,[45.0],J
1,physics_TD402,"Calculate the capacitance C of the capacitor, ...",100,45.0,[45.0],μF
2,physics_LD001,"Two charges, q1 = 6 × 10^-8 C and q2 = -6 × 10...",0.05,45.0,[45.0],N
